# LTDS to Powerflow Quality Control

This tutorial demonstrates a practical quality-control workflow for power flow time series by:

1. Selecting a substation from LTDS data
2. Pulling associated LTDS lines (Table 1) and transformers (Tables 2A/2B)
3. Cross-referencing with monthly powerflow circuits/transformers
4. Selecting a month with the least redaction
5. Running QC checks (gaps, anomalies, step changes, and flow balance)

**Prerequisites:**
- Complete [01-getting-started.ipynb](01-getting-started.ipynb) first
- Have your `UKPN_API_KEY` environment variable set
- These tutorials require additional dependencies. Install them with `pip install "ukpyn[all]"` — see [Tutorial 01](01-getting-started.ipynb) for full setup instructions
- complete `01-getting-started.ipynb` and `06-powerflow-timeseries.ipynb`.

In [1]:
import ukpyn

ukpyn.check_api_key()
print("API key configured!")

API key configured!


In [13]:
import pandas as pd

from ukpyn import ltds, powerflow
from ukpyn.utils import (
    detect_rolling_anomalies,
    detect_step_changes,
    fill_gaps,
    identify_gaps,
    quality_control,
    summarize_flow_balance,
    summarize_redaction_by_period,
)

try:
    import matplotlib.pyplot as plt

    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False

LICENCE_AREA = "Eastern Power Networks (EPN)"
REDaction_MARKERS = ["REDACTED", "SUPPRESSED", "MASKED"]
VALUE_FIELDS = [
    "active_power_mw",
    "reactive_power_mvar",
    "apparent_power_mva",
    "loading_percent",
]
print("Imports complete")

Imports complete


In [14]:
def records_to_dataframe(records: list[object]) -> pd.DataFrame:
    rows = []
    for record in records:
        if hasattr(record, "fields") and record.fields:
            rows.append(record.fields)
        elif isinstance(record, dict):
            rows.append(record)
    return pd.DataFrame(rows)


def substation_mask(df: pd.DataFrame, substation: str) -> pd.Series:
    if df.empty:
        return pd.Series([], dtype=bool)

    text_columns = [
        col
        for col in df.columns
        if pd.api.types.is_object_dtype(df[col])
        or pd.api.types.is_string_dtype(df[col])
    ]

    if not text_columns:
        return pd.Series(False, index=df.index)

    needle = str(substation).strip().lower()
    mask = pd.Series(False, index=df.index)
    for col in text_columns:
        mask = mask | df[col].astype("string").str.lower().str.contains(
            needle, na=False
        )
    return mask


def infer_rating_columns(df: pd.DataFrame) -> list[str]:
    if df.empty:
        return []
    candidates = []
    for col in df.columns:
        name = col.lower()
        if "rating" in name or "mva" in name:
            candidates.append(col)
    return candidates

## 1) Select a candidate substation from LTDS

We use LTDS Table 3A to discover a substation name that can then be cross-referenced into circuit and transformer datasets.

In [15]:
table_3a = ltds.get_table_3a(licence_area=LICENCE_AREA, limit=100)
print(table_3a)
df_3a = records_to_dataframe(table_3a.records)
print(df_3a.head())
if df_3a.empty or "substation" not in df_3a.columns:
    raise RuntimeError("Could not discover substations from LTDS table 3A")

substations = sorted(df_3a["substation"].dropna().astype(str).unique())
selected_substation = substations[0]

print(f"Selected substation: {selected_substation}")
print(f"Candidate substations found: {len(substations)}")

total_count=1110 links=None records=[Record(id=4, timestamp=None, size=None, fields={'gridsupplypoint': 'Amersham', 'substation': 'Coldharbour Farm 11kV', 'season': 'Summer', 'maximum_demand_24_25_mw': 2.9, 'maximum_demand_24_25_pf': 0.96, 'forecast_m_d_mw_25_26': 2.9, 'forecast_m_d_mw_26_27': 2.9, 'forecast_m_d_mw_27_28': 2.9, 'forecast_m_d_mw_28_29': 2.8, 'forecast_m_d_mw_29_30': 2.8, 'firm_capacity_mw': 11.5, 'minimum_load_scaling_factor': '28.8%', 'unutilised_capacity_percent': 74.78260869565217, 'functional_location': 'EPN-S0000000H8009', 'licencearea': 'Eastern Power Networks (EPN)'}, record_timestamp=None, links=None), Record(id=15, timestamp=None, size=None, fields={'gridsupplypoint': 'Barking 132kV', 'substation': 'Chase Cross Primary 11kV', 'season': 'Winter', 'maximum_demand_24_25_mw': 14.6, 'maximum_demand_24_25_pf': 0.96, 'forecast_m_d_mw_25_26': 14.8, 'forecast_m_d_mw_26_27': 15.6, 'forecast_m_d_mw_27_28': 15.8, 'forecast_m_d_mw_28_29': 16.2, 'forecast_m_d_mw_29_30': 16.8

In [18]:
table_1 = ltds.get_table_1(licence_area=LICENCE_AREA, limit=100)
table_2a = ltds.get_table_2a(
    licence_area=LICENCE_AREA, substation=selected_substation, limit=100
)
table_2b = ltds.get_table_2b(
    licence_area=LICENCE_AREA, substation=selected_substation, limit=100
)

df_table_1 = records_to_dataframe(table_1.records)
df_table_2a = records_to_dataframe(table_2a.records)
df_table_2b = records_to_dataframe(table_2b.records)

df_lines = df_table_1[substation_mask(df_table_1, selected_substation)].copy()
df_transformers = pd.concat([df_table_2a, df_table_2b], ignore_index=True)

line_rating_cols = infer_rating_columns(df_lines)
transformer_rating_cols = infer_rating_columns(df_transformers)

print(f"Associated LTDS lines: {len(df_lines)}")
print(f"Associated LTDS transformers: {len(df_transformers)}")
print(f"Line rating columns: {line_rating_cols[:6]}")
print(f"Transformer rating columns: {transformer_rating_cols[:6]}")

Associated LTDS lines: 0
Associated LTDS transformers: 2
Line rating columns: []
Transformer rating columns: ['transformer_rating_mva_winter', 'transformer_rating_mva_summer']


## 2) Pull monthly powerflow data and find least-redacted month

In [20]:
pf_132 = records_to_dataframe(
    powerflow.get_circuit_timeseries(
        voltage="132kv", granularity="monthly", licence_area=LICENCE_AREA, limit=100
    ).records
)
pf_33 = records_to_dataframe(
    powerflow.get_circuit_timeseries(
        voltage="33kv", granularity="monthly", licence_area=LICENCE_AREA, limit=100
    ).records
)
pf_grid = records_to_dataframe(
    powerflow.get_transformer_timeseries(
        transformer_type="grid",
        granularity="monthly",
        licence_area=LICENCE_AREA,
        limit=100,
    ).records
)
pf_primary = records_to_dataframe(
    powerflow.get_transformer_timeseries(
        transformer_type="primary",
        granularity="monthly",
        licence_area=LICENCE_AREA,
        limit=100,
    ).records
)

datasets = {
    "132kv_circuits": pf_132,
    "33kv_circuits": pf_33,
    "grid_transformers": pf_grid,
    "primary_transformers": pf_primary,
}

redaction_tables = []
for dataset_name, df in datasets.items():
    fields = [f for f in VALUE_FIELDS if f in df.columns]
    if not fields or "timestamp" not in df.columns:
        continue
    summary = summarize_redaction_by_period(
        df,
        timestamp_field="timestamp",
        value_fields=fields,
        markers=REDaction_MARKERS,
        include_missing_fallback=True,
    )
    summary["dataset"] = dataset_name
    redaction_tables.append(summary)

if not redaction_tables:
    raise RuntimeError(
        "No redaction summaries could be produced from fetched powerflow data"
    )

redaction_summary = pd.concat(redaction_tables, ignore_index=True)
monthly_redaction = redaction_summary.groupby("period", as_index=False)[
    ["total_points", "redacted_points"]
].sum()
monthly_redaction["redaction_rate"] = (
    monthly_redaction["redacted_points"] / monthly_redaction["total_points"]
)

best_month = monthly_redaction.sort_values("redaction_rate").iloc[0]["period"]
month_start = pd.Timestamp(best_month).normalize()
month_end = month_start + pd.offsets.MonthEnd(0)

print(f"Best month (least redaction): {month_start:%Y-%m}")
monthly_redaction.sort_values("redaction_rate").head()

RuntimeError: No redaction summaries could be produced from fetched powerflow data

## 3) Cross-reference IDs and run QC checks

In [ ]:
def filter_month(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty or "timestamp" not in df.columns:
        return df.copy()
    tmp = df.copy()
    tmp["timestamp"] = pd.to_datetime(tmp["timestamp"], errors="coerce")
    return tmp[
        (tmp["timestamp"] >= month_start) & (tmp["timestamp"] <= month_end)
    ].copy()


pf_132_m = filter_month(pf_132)
pf_33_m = filter_month(pf_33)
pf_grid_m = filter_month(pf_grid)
pf_primary_m = filter_month(pf_primary)

line_series_parts = []
for df in [pf_132_m, pf_33_m]:
    if {"timestamp", "active_power_mw"}.issubset(df.columns):
        s = (
            pd.to_numeric(df["active_power_mw"], errors="coerce")
            .groupby(pd.to_datetime(df["timestamp"], errors="coerce"))
            .sum(min_count=1)
        )
        line_series_parts.append(s)

transformer_series_parts = []
for df in [pf_grid_m, pf_primary_m]:
    if {"timestamp", "active_power_mw"}.issubset(df.columns):
        s = (
            pd.to_numeric(df["active_power_mw"], errors="coerce")
            .groupby(pd.to_datetime(df["timestamp"], errors="coerce"))
            .sum(min_count=1)
        )
        transformer_series_parts.append(s)

if not line_series_parts or not transformer_series_parts:
    raise RuntimeError(
        "Missing active_power_mw series for line/transformer balance checks"
    )

lines_total = pd.concat(line_series_parts, axis=1).sum(axis=1, min_count=1).sort_index()
transformers_total = (
    pd.concat(transformer_series_parts, axis=1).sum(axis=1, min_count=1).sort_index()
)

lines_qc = quality_control(lines_total, expected_frequency="1D")
line_gaps = identify_gaps(lines_total, expected_frequency="1D", min_gap_hours=24.0)
lines_filled = fill_gaps(lines_total, method="linear", max_gap_hours=72.0)
line_anomalies = detect_rolling_anomalies(lines_filled, window_size=3, threshold=5.0)
line_steps = detect_step_changes(
    lines_filled, threshold=0.1, window_size=2, min_confidence=0.5
)

balance_report = summarize_flow_balance(lines_filled, transformers_total, tolerance=0.2)

print(f"Line QC score: {lines_qc.quality_score:.1f}")
print(f"Line gaps detected: {len(line_gaps)}")
print(f"Line anomalies: {int(line_anomalies.sum())}")
print(f"Step changes: {len(line_steps)}")
balance_report

In [ ]:
line_ratings = df_lines[line_rating_cols].copy() if line_rating_cols else pd.DataFrame()
transformer_ratings = (
    df_transformers[transformer_rating_cols].copy()
    if transformer_rating_cols
    else pd.DataFrame()
)

print("Sample LTDS line ratings:")
display(
    line_ratings.head(10)
    if not line_ratings.empty
    else "No obvious line rating columns detected"
)

print("Sample LTDS transformer ratings:")
display(
    transformer_ratings.head(10)
    if not transformer_ratings.empty
    else "No obvious transformer rating columns detected"
)

## 4) Visualisations

In [ ]:
if not HAS_MATPLOTLIB:
    print("matplotlib not installed; skipping plots")
else:
    fig, axes = plt.subplots(3, 1, figsize=(12, 12))

    # Redaction trend by month
    axes[0].plot(
        monthly_redaction["period"], monthly_redaction["redaction_rate"], marker="o"
    )
    axes[0].set_title("Monthly redaction rate across powerflow datasets")
    axes[0].set_ylabel("Redaction rate")

    # Aggregate line vs transformer active power
    aligned = pd.concat(
        [lines_filled.rename("lines"), transformers_total.rename("transformers")],
        axis=1,
    ).dropna()
    axes[1].plot(aligned.index, aligned["lines"], label="Lines total MW")
    axes[1].plot(aligned.index, aligned["transformers"], label="Transformers total MW")
    axes[1].set_title("Aggregated line vs transformer active power")
    axes[1].legend()

    # Anomalies and step changes
    axes[2].plot(lines_filled.index, lines_filled.values, label="Lines total MW")
    anomaly_points = lines_filled[line_anomalies]
    if not anomaly_points.empty:
        axes[2].scatter(
            anomaly_points.index, anomaly_points.values, label="Anomalies", s=30
        )
    for step in line_steps:
        axes[2].axvline(step.timestamp, linestyle="--", alpha=0.5)
    axes[2].set_title("Line aggregate with anomalies and step changes")
    axes[2].legend()

    fig.tight_layout()
    plt.show()

## Summary

You now have a reproducible QC pattern that:

- Tracks candidate redaction by explicit markers with missingness fallback
- Selects a lower-redaction month before deeper diagnostics
- Compares aggregate line and transformer power totals
- Runs gap detection/filling, anomaly checks, and step-change checks

For project use, parameterise `LICENCE_AREA`, marker rules, and flow-balance tolerance for your governance standards.